In [1]:
import pandas as pd
import os
import glob

checkpoint_dir = "/workspace/llm-bias-evaluation/checkpoints"
cont_files = sorted(glob.glob(os.path.join(checkpoint_dir, "toxicity_cont_*.csv")))

print(f"{'File':<50} {'Total':>8} {'No Continuation':>16} {'%':>8} Status")
print("-" * 90)

for f in cont_files:
    df    = pd.read_csv(f)
    total = len(df)
    col   = df["continuation"].astype(str).str.strip()
    
    # Count ALL cases where no real continuation was generated
    no_cont = (
        df["continuation"].isna() |          # NaN float
        (col == "") |                         # empty string
        (col == "nan") |                      # string "nan"
        (col == ".") |                        # dot replacement
        (col.str.len() <= 1)                  # single character (meaningless)
    ).sum()
    
    pct    = (no_cont / total * 100) if total > 0 else 0
    fname  = os.path.basename(f)
    
    if pct == 0:
        status = "✓ Perfect"
    elif pct < 1:
        status = "⚠ minor (<1%) — acceptable"
    elif pct < 5:
        status = "⚠ moderate — acceptable"
    elif pct < 10:
        status = "✗ high — consider re-run"
    else:
        status = "✗ CRITICAL — must re-run"
    
    print(f"{fname:<50} {total:>8} {no_cont:>16} {pct:>7.2f}%  {status}")

File                                                  Total  No Continuation        % Status
------------------------------------------------------------------------------------------
toxicity_cont_BLOOM-560M_English.csv                  11143               26    0.23%  ⚠ minor (<1%) — acceptable
toxicity_cont_BLOOM-560M_Hindi.csv                    11143             5199   46.66%  ✗ CRITICAL — must re-run
toxicity_cont_BLOOM-560M_Tamil.csv                    11138             8250   74.07%  ✗ CRITICAL — must re-run
toxicity_cont_GPT-J-6B_English.csv                    11143                0    0.00%  ✓ Perfect
toxicity_cont_LLaMA-2-7B_English.csv                  11143                0    0.00%  ✓ Perfect
toxicity_cont_LLaMA-2-7B_Hindi.csv                    11143               75    0.67%  ⚠ minor (<1%) — acceptable
toxicity_cont_LLaMA-2-7B_Tamil.csv                    11138              177    1.59%  ⚠ moderate — acceptable
toxicity_cont_Mistral-7B_English.csv                  11143